# Exploratory Data Analysis for RAG Response Reliability

This notebook performs a reproducible exploratory analysis of the processed RAG reliability dataset.

## Goals

1. Load the train, validation, and test splits.
2. Inspect schema, missing values, and duplicates.
3. Detect the question, answer, context, and label columns.
4. Analyze Faithfulness, Relevancy, and Reliability labels.
5. Verify whether `Reliability = Faithfulness AND Relevancy`.
6. Analyze text lengths.
7. Check overlap between dataset splits.
8. Inspect representative error cases.
9. Save compact EDA reports under `results/data_exploration/`.

> Run the notebook from top to bottom.  
> If automatic column detection selects the wrong fields, edit the `MANUAL_COLUMNS` dictionary in Section 3.


## 1. Imports and display settings

In [ ]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 300)

RANDOM_STATE = 42


## 2. Locate the project and load all processed splits

In [ ]:
def find_project_root(start: Path) -> Path:
    """Find a parent directory containing the processed dataset directory."""
    start = start.resolve()

    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / "processed").is_dir():
            return candidate

    raise FileNotFoundError(
        "Could not find a project root containing a 'processed' directory. "
        "Save this notebook inside the repository and try again."
    )


ROOT_DIR = find_project_root(Path.cwd())
PROCESSED_DIR = ROOT_DIR / "processed"
RESULTS_DIR = ROOT_DIR / "results" / "data_exploration"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SPLIT_PATHS = {
    "train": PROCESSED_DIR / "train.csv",
    "validation": PROCESSED_DIR / "val.csv",
    "test": PROCESSED_DIR / "test.csv",
}

print("Project root:", ROOT_DIR)
print("Processed data:", PROCESSED_DIR)
print("EDA output:", RESULTS_DIR)


In [ ]:
datasets = {}

for split_name, file_path in SPLIT_PATHS.items():
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file: {file_path}")

    df = pd.read_csv(file_path, low_memory=False)
    datasets[split_name] = df

    print(
        f"{split_name:>10}: "
        f"{len(df):,} rows × {len(df.columns)} columns"
    )

train_df = datasets["train"]
validation_df = datasets["validation"]
test_df = datasets["test"]

display(train_df.head(3))


## 3. Inspect the schema and detect important columns

In [ ]:
schema_summary = pd.DataFrame({
    "column": train_df.columns,
    "dtype": train_df.dtypes.astype(str).values,
    "non_null_count": train_df.notna().sum().values,
    "missing_count": train_df.isna().sum().values,
    "missing_ratio": train_df.isna().mean().values,
    "unique_count": train_df.nunique(dropna=False).values,
})

display(schema_summary)
print("\nColumn names:")
for index, column in enumerate(train_df.columns, start=1):
    print(f"{index:>2}. {column}")


In [ ]:
# Edit only this dictionary if automatic detection chooses the wrong column.
# Keep a value as None to use automatic detection.

MANUAL_COLUMNS = {
    "question": None,
    "answer": None,
    "context": "chunk_1",  # primary chunk; full context built via build_combined_context() below
    "faithfulness": None,
    "relevancy": None,
    "reliability": None,
    "identifier": None,
}


CHUNK_COLUMNS = ["chunk_1", "chunk_2", "chunk_3", "chunk_4", "chunk_5"]


def build_combined_context(df: pd.DataFrame) -> pd.Series:
    """Concatenate all available chunk columns into a single context string.

    The processed dataset stores retrieved chunks as separate columns (chunk_1…chunk_5;
    chunk_6…chunk_8 may also be present for top-8 retrieval).  This helper safely joins
    all present chunk columns into one string per row so that text-length analysis and
    cross-split overlap checking can work on the complete context.

    Returns a Series of strings (NaN becomes an empty string).
    """
    available = [c for c in CHUNK_COLUMNS if c in df.columns]
    if not available:
        return pd.Series(dtype=str)
    joined = df[available].fillna("").astype(str).agg("\n\n".join, axis=1)
    return joined.str.strip()

COLUMN_ALIASES = {
    "question": [
        "question", "query", "user_query", "user_question",
        "prompt", "request", "user_message", "message"
    ],
    "answer": [
        "answer", "response", "generated_answer",
        "generated_response", "assistant_answer", "assistant_response"
    ],
    "context": [
        "context", "contexts", "retrieved_context",
        "retrieved_contexts", "retrieved_chunks", "chunks",
        "documents", "source_documents", "knowledge"
    ],
    "faithfulness": [
        "faithfulness", "faithful", "groundedness",
        "grounded", "is_faithful"
    ],
    "relevancy": [
        "relevancy", "relevance", "relevant",
        "answer_relevancy", "is_relevant"
    ],
    "reliability": [
        "reliability", "reliable", "joint_reliability",
        "joint_label", "is_reliable"
    ],
    "identifier": [
        "id", "sample_id", "row_id", "conversation_id",
        "dialog_id", "chat_id", "request_id"
    ],
}


def normalize_column_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")


def detect_column(columns, aliases):
    """Prefer exact normalized matches, then controlled substring matches."""
    normalized_map = {
        normalize_column_name(column): column
        for column in columns
    }

    for alias in aliases:
        normalized_alias = normalize_column_name(alias)
        if normalized_alias in normalized_map:
            return normalized_map[normalized_alias]

    for alias in aliases:
        normalized_alias = normalize_column_name(alias)
        matches = [
            original
            for normalized, original in normalized_map.items()
            if normalized_alias in normalized
        ]
        if len(matches) == 1:
            return matches[0]

    return None


SELECTED_COLUMNS = {}

for role, aliases in COLUMN_ALIASES.items():
    manual_value = MANUAL_COLUMNS.get(role)

    if manual_value is not None:
        if manual_value not in train_df.columns:
            raise KeyError(
                f"Manual column '{manual_value}' for role '{role}' "
                "does not exist in the training data."
            )
        SELECTED_COLUMNS[role] = manual_value
    else:
        SELECTED_COLUMNS[role] = detect_column(train_df.columns, aliases)

selected_columns_table = pd.DataFrame(
    [
        {"role": role, "selected_column": column}
        for role, column in SELECTED_COLUMNS.items()
    ]
)

display(selected_columns_table)

unresolved_roles = [
    role for role, column in SELECTED_COLUMNS.items()
    if column is None
]

if unresolved_roles:
    print(
        "Unresolved roles:",
        unresolved_roles,
        "\nThis is not always an error. Edit MANUAL_COLUMNS only when needed."
    )


## 4. Dataset size, missing values, and duplicates

In [ ]:
split_summary_records = []

for split_name, df in datasets.items():
    split_summary_records.append({
        "split": split_name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicate_rows": int(df.duplicated().sum()),
        "exact_duplicate_ratio": float(df.duplicated().mean()),
    })

split_summary = pd.DataFrame(split_summary_records)
display(split_summary)

split_summary.to_csv(
    RESULTS_DIR / "split_overview.csv",
    index=False,
)


In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(split_summary["split"], split_summary["rows"])
plt.title("Number of Samples by Dataset Split")
plt.xlabel("Dataset Split")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()


In [ ]:
missing_reports = []

for split_name, df in datasets.items():
    report = pd.DataFrame({
        "split": split_name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_ratio": df.isna().mean().values,
    })
    missing_reports.append(report)

missing_report = pd.concat(missing_reports, ignore_index=True)
missing_report = missing_report.sort_values(
    ["missing_ratio", "missing_count"],
    ascending=False,
)

display(missing_report[missing_report["missing_count"] > 0].head(50))

missing_report.to_csv(
    RESULTS_DIR / "missing_values.csv",
    index=False,
)


## 5. Normalize and analyze label columns

In [ ]:
def normalize_binary_label(series: pd.Series) -> pd.Series:
    """Convert common binary label formats to pandas nullable integers.

    Note: This function does NOT handle joint labels like "0_1", "1_0", etc.
    Use normalize_reliability_label() for joint_label columns instead.
    """
    truthy = {
        "1", "true", "yes", "y", "positive",
        "faithful", "relevant", "reliable", "supported"
    }
    falsy = {
        "0", "false", "no", "n", "negative",
        "unfaithful", "irrelevant", "unreliable", "unsupported"
    }

    def convert(value):
        if pd.isna(value):
            return pd.NA

        if isinstance(value, (bool, np.bool_)):
            return int(value)

        if isinstance(value, (int, np.integer)):
            return int(value) if value in (0, 1) else pd.NA

        if isinstance(value, (float, np.floating)):
            if value in (0.0, 1.0):
                return int(value)
            return pd.NA

        normalized = str(value).strip().lower()

        if normalized in truthy:
            return 1
        if normalized in falsy:
            return 0

        # Explicitly reject strings with underscores to prevent
        # mis-parsing of joint labels like "0_1" as numeric 0 or 1.
        if "_" in normalized:
            return pd.NA

        try:
            numeric = float(normalized)
            if numeric in (0.0, 1.0):
                return int(numeric)
        except ValueError:
            pass

        return pd.NA

    return series.map(convert).astype("Int64")


def normalize_reliability_label(series: pd.Series) -> pd.Series:
    """Normalize joint labels for reliability = (relevancy AND faithfulness).

    joint_label format: {relevancy}_{faithfulness}
    Examples:
        - "1_1" -> 1  (reliable: both relevancy and faithfulness are true)
        - "0_0" -> 0  (unreliable: both are false)
        - "0_1" -> 0  (unreliable: relevancy is false)
        - "1_0" -> 0  (unreliable: faithfulness is false)

    Returns a pandas nullable integer series (0, 1, or NA).
    """
    RELIABILITY_MAPPING = {
        "1_1": 1,  # reliable: relevancy=1 AND faithfulness=1
        "0_0": 0,  # unreliable: relevancy=0 AND faithfulness=0
        "0_1": 0,  # unreliable: relevancy=0
        "1_0": 0,  # unreliable: faithfulness=0
    }

    def convert(value):
        if pd.isna(value):
            return pd.NA

        normalized = str(value).strip()
        return RELIABILITY_MAPPING.get(normalized, pd.NA)

    return series.map(convert).astype("Int64")


LABEL_ROLES = ["faithfulness", "relevancy"]
NORMALIZED_LABEL_COLUMNS = {}

for split_name, df in datasets.items():
    for role in LABEL_ROLES:
        source_column = SELECTED_COLUMNS.get(role)

        if source_column is None or source_column not in df.columns:
            continue

        normalized_column = f"__{role}_binary"
        df[normalized_column] = normalize_binary_label(df[source_column])
        NORMALIZED_LABEL_COLUMNS[role] = normalized_column

# Handle reliability separately using joint_label with explicit mapping
reliability_role = "reliability"
reliability_source = SELECTED_COLUMNS.get(reliability_role)
if reliability_source:
    for split_name, df in datasets.items():
        if reliability_source in df.columns:
            normalized_column = f"__{reliability_role}_binary"
            df[normalized_column] = normalize_reliability_label(df[reliability_source])
            NORMALIZED_LABEL_COLUMNS[reliability_role] = normalized_column

print("Normalized label columns:", NORMALIZED_LABEL_COLUMNS)


In [ ]:
label_distribution_records = []

for split_name, df in datasets.items():
    for role, normalized_column in NORMALIZED_LABEL_COLUMNS.items():
        counts = df[normalized_column].value_counts(dropna=False)

        for label_value, count in counts.items():
            label_distribution_records.append({
                "split": split_name,
                "label": role,
                "value": str(label_value),
                "count": int(count),
                "ratio": float(count / len(df)),
            })

label_distribution = pd.DataFrame(label_distribution_records)
display(label_distribution)

label_distribution.to_csv(
    RESULTS_DIR / "label_distribution.csv",
    index=False,
)


In [ ]:
for role, normalized_column in NORMALIZED_LABEL_COLUMNS.items():
    counts = (
        train_df[normalized_column]
        .value_counts(dropna=False)
        .sort_index()
    )

    plt.figure(figsize=(6, 4))
    counts.plot(kind="bar")
    plt.title(f"{role.title()} Label Distribution in Training Data")
    plt.xlabel("Label Value")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()


## 6. Joint Faithfulness–Relevancy analysis

In [ ]:
faith_col = NORMALIZED_LABEL_COLUMNS.get("faithfulness")
relev_col = NORMALIZED_LABEL_COLUMNS.get("relevancy")
reliability_col = NORMALIZED_LABEL_COLUMNS.get("reliability")

if faith_col and relev_col:
    joint_counts = pd.crosstab(
        train_df[faith_col],
        train_df[relev_col],
        margins=True,
        dropna=False,
    )

    joint_ratios = pd.crosstab(
        train_df[faith_col],
        train_df[relev_col],
        normalize="all",
        dropna=False,
    )

    print("Joint counts:")
    display(joint_counts)

    print("Joint ratios:")
    display(joint_ratios)
else:
    print(
        "Faithfulness and/or Relevancy columns were not detected. "
        "Edit MANUAL_COLUMNS and rerun the notebook."
    )


In [ ]:
reliability_mismatch_report = pd.DataFrame()

if faith_col and relev_col:
    valid_mask = (
        train_df[faith_col].notna()
        & train_df[relev_col].notna()
    )

    train_df.loc[valid_mask, "__expected_reliability"] = (
        train_df.loc[valid_mask, faith_col].astype(int)
        & train_df.loc[valid_mask, relev_col].astype(int)
    )

    train_df["__expected_reliability"] = (
        train_df["__expected_reliability"].astype("Int64")
    )

    if reliability_col:
        comparison_mask = (
            valid_mask
            & train_df[reliability_col].notna()
        )

        mismatch_mask = (
            comparison_mask
            & (
                train_df["__expected_reliability"]
                != train_df[reliability_col]
            )
        )

        print(
            "Reliability label mismatches:",
            int(mismatch_mask.sum())
        )

        report_columns = [
            column for column in [
                SELECTED_COLUMNS.get("identifier"),
                SELECTED_COLUMNS.get("faithfulness"),
                SELECTED_COLUMNS.get("relevancy"),
                SELECTED_COLUMNS.get("reliability"),
                faith_col,
                relev_col,
                reliability_col,
                "__expected_reliability",
            ]
            if column is not None and column in train_df.columns
        ]

        reliability_mismatch_report = train_df.loc[
            mismatch_mask,
            report_columns,
        ].copy()

        display(reliability_mismatch_report.head(20))

        reliability_mismatch_report.to_csv(
            RESULTS_DIR / "reliability_label_mismatches.csv",
            index=False,
        )
    else:
        print(
            "No explicit Reliability column was detected. "
            "The expected joint Reliability label was calculated internally."
        )


## 7. Text-length analysis

In [ ]:
TEXT_ROLES = ["question", "answer", "context"]
text_length_records = []

for split_name, df in datasets.items():
    for role in TEXT_ROLES:
        column = SELECTED_COLUMNS.get(role)

        if column is None or column not in df.columns:
            continue

        # For the context role, always use the combined all-chunks version
        # so that text-length analysis reflects the full retrieved context.
        if role == "context":
            text = build_combined_context(df)
        else:
            text = df[column].fillna("").astype(str)

        char_length = text.str.len()
        whitespace_token_length = text.str.split().str.len()

        text_length_records.append({
            "split": split_name,
            "role": role,
            "column": column,
            "mean_characters": float(char_length.mean()),
            "median_characters": float(char_length.median()),
            "p95_characters": float(char_length.quantile(0.95)),
            "p99_characters": float(char_length.quantile(0.99)),
            "max_characters": int(char_length.max()),
            "mean_whitespace_tokens": float(whitespace_token_length.mean()),
            "median_whitespace_tokens": float(whitespace_token_length.median()),
            "p95_whitespace_tokens": float(
                whitespace_token_length.quantile(0.95)
            ),
            "max_whitespace_tokens": int(whitespace_token_length.max()),
        })

text_length_summary = pd.DataFrame(text_length_records)
display(text_length_summary)

text_length_summary.to_csv(
    RESULTS_DIR / "text_length_summary.csv",
    index=False,
)


In [ ]:
for role in TEXT_ROLES:
    column = SELECTED_COLUMNS.get(role)

    if column is None or column not in train_df.columns:
        continue

    lengths = train_df[column].fillna("").astype(str).str.len()
    upper_limit = lengths.quantile(0.99)
    filtered_lengths = lengths[lengths <= upper_limit]

    plt.figure(figsize=(8, 4))
    plt.hist(filtered_lengths, bins=50)
    plt.title(
        f"{role.title()} Length Distribution "
        "(Up to the 99th Percentile)"
    )
    plt.xlabel("Number of Characters")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()


## 8. Check overlap between train, validation, and test

In [ ]:
CONTEXT_COL = SELECTED_COLUMNS.get("context")

if CONTEXT_COL:
    datasets_with_context = {}
    for split_name, df in datasets.items():
        df_copy = df.copy()
        df_copy["__combined_context"] = build_combined_context(df)
        datasets_with_context[split_name] = df_copy
else:
    datasets_with_context = datasets

signature_columns = [
    SELECTED_COLUMNS.get("question"),
    SELECTED_COLUMNS.get("answer"),
    # use the derived combined-context column instead of the raw chunk column
    "__combined_context" if CONTEXT_COL else None,
]

signature_columns = [
    column
    for column in signature_columns
    if column is not None
    and all(column in df.columns for df in datasets_with_context.values())
]

print("Columns used for overlap checking:", signature_columns)


def create_row_signatures(df: pd.DataFrame, columns) -> set:
    if not columns:
        return set()

    normalized = df[columns].fillna("").astype(str).copy()

    for column in columns:
        normalized[column] = (
            normalized[column]
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    hashes = pd.util.hash_pandas_object(
        normalized,
        index=False,
    )

    return set(hashes.tolist())


if signature_columns:
    split_signatures = {
        split_name: create_row_signatures(df, signature_columns)
        for split_name, df in datasets_with_context.items()
    }

    split_names = list(split_signatures)
    overlap_records = []

    for first_index, first_split in enumerate(split_names):
        for second_split in split_names[first_index + 1:]:
            overlap = (
                split_signatures[first_split]
                & split_signatures[second_split]
            )

            overlap_records.append({
                "first_split": first_split,
                "second_split": second_split,
                "overlap_count": len(overlap),
            })

    overlap_summary = pd.DataFrame(overlap_records)
    display(overlap_summary)

    overlap_summary.to_csv(
        RESULTS_DIR / "cross_split_overlap.csv",
        index=False,
    )
else:
    print(
        "No suitable text columns were detected for overlap checking. "
        "Edit MANUAL_COLUMNS and rerun."
    )


## 9. Inspect representative error cases

In [ ]:
inspection_columns = [
    SELECTED_COLUMNS.get("identifier"),
    SELECTED_COLUMNS.get("question"),
    SELECTED_COLUMNS.get("answer"),
    SELECTED_COLUMNS.get("context"),
    SELECTED_COLUMNS.get("faithfulness"),
    SELECTED_COLUMNS.get("relevancy"),
    SELECTED_COLUMNS.get("reliability"),
]

inspection_columns = [
    column
    for column in inspection_columns
    if column is not None and column in train_df.columns
]

print("Inspection columns:", inspection_columns)


In [ ]:
def display_random_examples(
    df: pd.DataFrame,
    mask: pd.Series,
    title: str,
    columns,
    n: int = 5,
):
    subset = df.loc[mask, columns]

    print(f"{title}: {len(subset):,} samples")

    if subset.empty:
        return

    display(
        subset.sample(
            n=min(n, len(subset)),
            random_state=RANDOM_STATE,
        )
    )


if faith_col:
    display_random_examples(
        train_df,
        train_df[faith_col] == 0,
        "Unfaithful examples",
        inspection_columns,
    )

if relev_col:
    display_random_examples(
        train_df,
        train_df[relev_col] == 0,
        "Irrelevant examples",
        inspection_columns,
    )

if faith_col and relev_col:
    display_random_examples(
        train_df,
        (train_df[faith_col] == 0)
        & (train_df[relev_col] == 1),
        "Relevant but unfaithful examples",
        inspection_columns,
    )

    display_random_examples(
        train_df,
        (train_df[faith_col] == 1)
        & (train_df[relev_col] == 0),
        "Faithful but irrelevant examples",
        inspection_columns,
    )


## 10. Read existing project quality reports

In [ ]:
quality_report_path = PROCESSED_DIR / "data_quality_report.json"
existing_split_summary_path = PROCESSED_DIR / "split_summary.csv"

if quality_report_path.exists():
    with quality_report_path.open("r", encoding="utf-8") as file:
        quality_report = json.load(file)

    print("Existing data quality report:")
    display(quality_report)
else:
    print("No data_quality_report.json file was found.")

if existing_split_summary_path.exists():
    print("\nExisting split summary:")
    existing_split_summary = pd.read_csv(
        existing_split_summary_path,
        low_memory=False,
    )
    display(existing_split_summary)
else:
    print("No split_summary.csv file was found.")


## 11. Export a compact JSON summary

In [ ]:
eda_summary = {
    "selected_columns": SELECTED_COLUMNS,
    "split_sizes": {
        split_name: int(len(df))
        for split_name, df in datasets.items()
    },
    "exact_duplicate_rows": {
        split_name: int(df.duplicated().sum())
        for split_name, df in datasets.items()
    },
    "label_columns_detected": list(NORMALIZED_LABEL_COLUMNS.keys()),
    "signature_columns": signature_columns,
}

# Document how the context field is handled since it spans multiple chunk columns.
context_col = SELECTED_COLUMNS.get("context")
if context_col is None:
    eda_summary["context_note"] = (
        "The processed dataset does not contain a retrieved-context column, "
        "so context-length analysis could not be performed."
    )
else:
    eda_summary["context_note"] = (
        f"The context is stored across multiple chunk columns "
        f"(chunk_1 … chunk_5; chunk_6–8 optional). "
        f"MANUAL_COLUMNS['context'] = '{context_col}' was used as the primary identifier. "
        f"The build_combined_context() helper concatenates all available chunks "
        f"for text-length analysis and cross-split overlap checking."
    )

if faith_col and relev_col:
    valid_joint = train_df[
        train_df[faith_col].notna()
        & train_df[relev_col].notna()
    ]

    eda_summary["train_joint_label_counts"] = {
        f"faithfulness_{int(faith)}_relevancy_{int(relevancy)}": int(count)
        for (faith, relevancy), count in (
            valid_joint
            .groupby([faith_col, relev_col])
            .size()
            .items()
        )
    }

if faith_col and relev_col and reliability_col:
    eda_summary["reliability_label_mismatch_count"] = int(
        len(reliability_mismatch_report)
    )

summary_path = RESULTS_DIR / "eda_summary.json"

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(
        eda_summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:", summary_path)
display(eda_summary)


## 12. Main findings

Complete this section after running all cells.

### Dataset structure

- Train samples:
- Validation samples:
- Test samples:
- Question column:
- Answer column:
- Context column: **Note:** The processed dataset stores retrieved context across multiple columns (`chunk_1` … `chunk_5`, with `chunk_6`–`chunk_8` optional). `chunk_1` was selected as the primary identifier. The `build_combined_context()` helper concatenates all available chunks for text-length analysis and cross-split overlap checking.
- Label columns:

### Data quality

- Missing-value issues:
- Exact duplicates:
- Cross-split overlap:
- Reliability-label inconsistencies:

### Label balance

- Faithfulness positive ratio:
- Relevancy positive ratio:
- Reliability positive ratio:
- Most common Faithfulness–Relevancy combination:

### Text characteristics

- Median question length:
- Median answer length:
- Median context length:
- Important long-text outliers:

### Implications for modeling

- Is class weighting needed?
- Why should macro-F1 be reported?
- Which subgroup appears most difficult?
- Does context length affect the choice of model?
- Recommended next experiment:
